In [67]:
import numpy as np
import pandas as pd

# from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score

from claimlens.data import load_data


In [68]:
df = pd.read_parquet('../data/processed/abcd_first_msgs.parquet')
# df_sample = pd.read_json('../raw_data/abcd_sample.json')


In [69]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me with that?",0,1,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to know when they expire. I'd like to use it to buy some hats for my cat.,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deatails as they have changed recently due to a move",0,1,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. And I need some new clothes ASAP. I am thinking of ordering them to come by overnight shipping Do you know how much that costs?,0,0,34
...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,HI I want to upgrade an order I have with you to overnight shipping,0,0,14
10038,test,3293,manage_account,status_credit_missing,"Hi, I think I am missing a $40 credit from a subscription refund.",0,0,13
10039,test,8492,subscription_inquiry,manage_extension,I would like an extension on my subscription.,0,0,8
10040,test,189,storewide_query,pricing_1,Hi! How much do you charge to gift wrap items?,0,0,10


In [53]:
# pd.set_option("display.max_colwidth", None)   # show full text in tables (run once)

# def show_claims(subflow, n=20):
#     """Show up to n claims for one subflow, with their refund label."""
#     rows = df_full[df_full["subflow"] == subflow]       # all chats with this subflow

#     refund_rate = rows["refund"].mean()                 # share of chats that were refunded
#     print(f"{subflow}: {len(rows)} chats, {refund_rate:.0%} refunded")

#     return rows[["first_msgs", "refund"]].head(n)       # table with just the text and the label

# show_claims("return_size")

In [54]:
# REFUND_SUBFLOWS = {
#     "refund_initiate",
#     "return_size",
#     "return_color",
#     "return_stain",
#     "manage_dispute_bill",
#     "bad_price_competitor",
#     "status_mystery_fee",
#     "refund_update",
#     "manage_cancel",
# }

# df["refund_request"] = df["subflow"].isin(REFUND_SUBFLOWS).astype(int)
# print(df["refund_request"].value_counts())

In [ ]:
TARGET = "refund_request"

      split  claim_id               flow                subflow  \
0     train      3592     product_defect            return_size   
1     train      9489     product_defect          refund_status   
2     train      3695    storewide_query               timing_4   
3     train      5798     shipping_issue                 manage   
4     train      3647    storewide_query              pricing_3   
...     ...       ...                ...                    ...   
8029  train      2229  troubleshoot_site         search_results   
8030  train      6109     manage_account  status_credit_missing   
8031  train      9892  single_item_query            jeans_how_2   
8032  train      4036  single_item_query          jeans_other_2   
8033  train      5882  troubleshoot_site         search_results   

                                                                                                                                                                                                   

In [56]:
X_train = train['cleaned_text']
y_train = train['refund_request']

X_val = val['cleaned_text']
y_val = val['refund_request']

In [57]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
}

In [58]:
for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    print(f"\n=== {name} ===")
    print(classification_report(y_val, pipe.predict(X_val)))


=== count + none ===
              precision    recall  f1-score   support

           0       0.95      0.97      0.96       917
           1       0.61      0.41      0.49        87

    accuracy                           0.93      1004
   macro avg       0.78      0.69      0.73      1004
weighted avg       0.92      0.93      0.92      1004


=== count + balanced ===
              precision    recall  f1-score   support

           0       0.98      0.90      0.94       917
           1       0.45      0.85      0.59        87

    accuracy                           0.90      1004
   macro avg       0.72      0.88      0.77      1004
weighted avg       0.94      0.90      0.91      1004


=== tfidf + none ===
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       917
           1       0.74      0.32      0.45        87

    accuracy                           0.93      1004
   macro avg       0.84      0.66      0.71      1004
weig

In [59]:
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)

    results.append({
        "model": name,
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred)
    })
    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)


In [60]:
results_df


,model,precision,recall,f1
3,tfidf + balanced,0.456140,0.896552,0.604651
1,count + balanced,0.453988,0.850575,0.592000
0,count + none,0.610169,0.413793,0.493151
2,tfidf + none,0.736842,0.321839,0.448000


In [61]:
best_name = results_df.iloc[0]["model"]
model = models[best_name]

print("Best model:", best_name)

Best model: tfidf + balanced


In [62]:
# df_full['flow'] = df_full['scenario'].str['flow']
# df_full['subflow'] = df_full['scenario'].str['subflow']

In [63]:
# def quick_claim_text(turns, n=3):
#     """Rough version: first n customer messages joined. Temporary until the group's cleaned column is ready."""
#     msgs = [t["text"] for t in turns if t["speaker"] == "customer"]   # customer turns only
#     return " ".join(msgs[:n])                                         # first n, joined with spaces

# df_full["claim_text"] = df_full["delexed"].apply(quick_claim_text)
# df_full["subflow"] = df_full["scenario"].str["subflow"]

In [64]:
# '''Checking if Dataframe is received correctly'''
# REQUIRED = ["claim_id", "claim_text", "subflow", "wants_refund", "split"]
# missing = set(REQUIRED) - set(df.columns)
# assert not missing, f"Missing columns: {missing}"

# assert df["claim_id"].is_unique, "Duplicate claim IDs"

# df["wants_refund"] = Rdf["wants_refund"].astype(int)
# print("empty texts:", (df["claim_text"].fillna("").str.strip() == "").sum())
# print("missing subflows:", df["subflow"].isna().sum())
# print(df["split"].value_counts())

In [65]:
# REFUND_SUBFLOWS = set(df.loc[df["wants_refund"] == 1, "subflow"])
# sorted(REFUND_SUBFLOWS)

In [66]:
# df_full["subflow"].value_counts()